# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haideriqbal499/Flyrank_Ml_internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.**  
Turn the validated ranking into a human-reviewed content action playbook: ranked actions,
reason codes, archetype→action map, decay/refresh limits, cost/value, monitoring triggers,
and a clear no-go list. Practical and non-production.

> Skills: `writing-honest-claims` + `flyrank/flyrank-data`.

## 1. Ranked actions + reason codes

A model score alone is not the product. Editors need: **what to do first**, **why** (reason codes),
and **which page type** (archetype) they are looking at.

### How the queue is built (this notebook)

1. **Model rank** — Random Forest decline probability on the same honest feature set as Week 5
   (no `trend_*` / label columns in X). Trained on client-holdout train clients; scores applied
   to the full inventory for a reviewer queue.
2. **Transparent baseline blend** — visibility / freshness / position / depth scores (starter formula).
3. **Final score** — `100 * (0.70 * model_prob + 0.30 * normalized_baseline)`.
4. **Reason codes** — short tags a human can challenge (rules + model thresholds).
5. **Suggested action** — mapped from reasons, not from a black box alone.

### Decay / refresh insight (from Week 4 signal audit)

| Signal | Verdict on this starter snapshot | Playbook consequence |
|---|---|---|
| Staleness (`days_since_last_update` deep-stale vs fresh) | **OPPOSITE** | Do **not** auto-prioritize solely because a page is old. Age is context, not a green light. |
| Low CTR among visible pages (imp≥500, pos 1–20, ctr&lt;0.5) | **CONFIRMED** | Prefer CTR-review actions on high-demand visible pages. |

### Archetype → action map (decision support)

| Archetype | Typical evidence | Suggested action |
|---|---|---|
| `visible_ctr_gap` | High impressions, page 1–2, low CTR | `refresh_and_review_ctr` |
| `engagement_gap` | Enough sessions, weak engagement/scroll | `refresh_and_review_engagement` |
| `thin_visible` | Visible demand + thin word count | `expand_and_refresh` |
| `high_demand_decline_risk` | Model/flag risk + meaningful impressions | `refresh` |
| `page_one_aged` | Page-one + older content (context only) | `refresh` *after* demand check — not age alone |
| `low_demand_noise` | Little search demand | `monitor` (or prune only after human judgment) |
| `strong_visible` | High demand, healthy CTR/engagement, low model risk | `protect` (do not churn) |

Honest framing: these are **observed** triage tags on one anonymized snapshot. They are
**directional decision-support**, not proof that an edit recovers traffic.

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/haideriqbal499/Flyrank_Ml_internship"
REPO_DIR = "Flyrank_Ml_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.abspath(os.sep):
        os.chdir("..")

CSV = Path("data/raw/content_refresh_anonymized.csv")
OUT_DIR = Path("work/outputs")
FIG_DIR = Path("work/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

QUEUE_PATH = OUT_DIR / "w07_action_playbook_queue.csv"
METRICS_PATH = OUT_DIR / "w07_action_playbook_metrics.json"
W05_PATH = OUT_DIR / "w05_model_vs_baseline.json"
W06_PATH = OUT_DIR / "w06_validation_audit.json"

df = pd.read_csv(CSV)
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
df["word_count"] = pd.to_numeric(df["word_count"], errors="coerce").fillna(0)

# --- features (same honest set as Week 5; no trend / label leakage) ---
df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"].clip(lower=0))
df["log_sessions_90d"] = np.log1p(df["sessions_90d"].clip(lower=0))
df["has_position"] = (df["avg_position"] > 0).astype(int)
df["has_word_count"] = (df["word_count"] > 0).astype(int)
df["word_count_filled"] = df["word_count"]

NUMERIC_FEATURES = [
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "has_position",
    "engagement_rate",
    "scroll_rate",
    "word_count_filled",
    "has_word_count",
]

X = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].astype(int)
clients = df["client_id"].fillna("unknown").astype(str)

unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = clients.isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

rf = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])
prob_all = rf.predict_proba(X)[:, 1]
prob_test = prob_all[test_idx]
y_test = y.iloc[test_idx].to_numpy()

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean()) if len(order) else 0.0

p10 = precision_at_k(y_test, prob_test, 10)
p50 = precision_at_k(y_test, prob_test, 50)
roc = float(roc_auc_score(y_test, prob_test))
ap = float(average_precision_score(y_test, prob_test))

# --- transparent baseline components ---
def percentile_rank(s):
    return pd.to_numeric(s, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def normalize(s):
    v = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    lo, hi = float(v.min()), float(v.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.zeros(len(v)), index=v.index)
    return (v - lo) / (hi - lo)

visibility = percentile_rank(np.log1p(df["impressions_90d"]))
freshness_risk = percentile_rank(df["days_since_last_update"])
position_opp = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * visibility
    * (df["avg_position"] > 0).astype(int)
)
depth_gap = (1 - percentile_rank(df["word_count"])) * visibility
baseline = (0.40 * visibility + 0.30 * freshness_risk + 0.25 * position_opp + 0.05 * depth_gap).clip(0, 1)

queue = df.copy()
queue["model_probability"] = prob_all
queue["baseline_refresh_score"] = baseline
queue["final_refresh_score"] = (
    100 * (0.70 * queue["model_probability"] + 0.30 * normalize(queue["baseline_refresh_score"]))
).clip(0, 100)

def reason_codes(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if str(row["trend_direction"]).lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if row["avg_position"] > 0 and row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("low_engagement_visible_page")
    if row["model_probability"] >= 0.65:
        reasons.append("model_decline_risk")
    if row["model_probability"] >= 0.50 and row["impressions_90d"] >= 500:
        reasons.append("visible_model_opportunity")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("ctr_review_candidate")
    if row["sessions_90d"] >= 30 and (
        (row["engagement_rate"] > 0 and row["engagement_rate"] < 30)
        or (row["scroll_rate"] > 0 and row["scroll_rate"] < 30)
    ):
        reasons.append("engagement_review_candidate")
    # de-dupe preserve order
    out = []
    for r in reasons:
        if r not in out:
            out.append(r)
    return "|".join(out or ["general_refresh_review"])

queue["reason_codes"] = queue.apply(reason_codes, axis=1)

def suggested_action(row):
    reasons = set(str(row["reason_codes"]).split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "ctr_review_candidate" in reasons and (
        "model_decline_risk" in reasons or "declining_with_demand" in reasons
    ):
        return "refresh_and_review_ctr"
    if "engagement_review_candidate" in reasons and (
        "model_decline_risk" in reasons or "declining_with_demand" in reasons
    ):
        return "refresh_and_review_engagement"
    if {
        "model_decline_risk",
        "declining_with_demand",
        "stale_visible_page",
        "visible_model_opportunity",
        "page_one_decay_risk",
    }.intersection(reasons):
        return "refresh"
    # protect strong visible pages with low model risk
    if (
        row["impressions_90d"] >= 1000
        and row["model_probability"] < 0.40
        and row["ctr"] >= 0.5
        and 0 < row["avg_position"] <= 10
    ):
        return "protect"
    return "monitor"

def assign_archetype(row):
    reasons = set(str(row["reason_codes"]).split("|"))
    if "thin_visible_page" in reasons:
        return "thin_visible"
    if "ctr_review_candidate" in reasons or "low_ctr_visible_page" in reasons:
        return "visible_ctr_gap"
    if "engagement_review_candidate" in reasons or "low_engagement_visible_page" in reasons:
        return "engagement_gap"
    if row["impressions_90d"] < 100:
        return "low_demand_noise"
    if "page_one_decay_risk" in reasons:
        return "page_one_aged"
    if {
        "model_decline_risk",
        "declining_with_demand",
        "visible_model_opportunity",
    }.intersection(reasons):
        return "high_demand_decline_risk"
    if row["suggested_action"] == "protect":
        return "strong_visible"
    return "general_review"

queue["suggested_action"] = queue.apply(suggested_action, axis=1)
queue["archetype"] = queue.apply(assign_archetype, axis=1)

high_thr = float(queue["final_refresh_score"].quantile(0.8))
med_thr = float(queue["final_refresh_score"].quantile(0.5))

def confidence_label(row):
    if (
        row["final_refresh_score"] >= high_thr
        and row["impressions_90d"] >= 500
        and row["sessions_90d"] >= 10
        and row["model_probability"] >= 0.5
    ):
        return "high"
    if row["final_refresh_score"] >= med_thr:
        return "medium"
    return "low"

queue["confidence"] = queue.apply(confidence_label, axis=1)
queue = queue.sort_values(
    ["final_refresh_score", "impressions_90d", "sessions_90d"],
    ascending=[False, False, False],
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

print("Working dir:", os.getcwd())
print(f"Rows scored: {len(queue):,} | declining-label rate: {y.mean():.3f}")
print(
    f"Client-holdout check (train→test): Precision@10={p10:.3f} Precision@50={p50:.3f} "
    f"ROC AUC={roc:.3f} AP={ap:.3f} | test n={len(test_idx):,} clients={n_test_clients}"
)
print("\nAction mix:")
print(queue["suggested_action"].value_counts().to_string())
print("\nArchetype mix:")
print(queue["archetype"].value_counts().to_string())
print("\nTop 10 preview:")
print(
    queue.head(10)[
        [
            "rank",
            "final_refresh_score",
            "suggested_action",
            "archetype",
            "confidence",
            "reason_codes",
            "impressions_90d",
            "avg_position",
            "ctr",
        ]
    ].to_string(index=False)
)

Working dir: C:\Users\Windows 11\Flyrank_Ml_internship
Rows scored: 30,000 | declining-label rate: 0.542
Client-holdout check (train→test): Precision@10=0.900 Precision@50=0.900 ROC AUC=0.764 AP=0.668 | test n=2,325 clients=6

Action mix:
suggested_action
refresh                          11717
monitor                           9725
refresh_and_review_ctr            6296
refresh_and_review_engagement     1879
protect                            301
expand_and_refresh                  82

Archetype mix:
archetype
visible_ctr_gap             9741
low_demand_noise            7953
high_demand_decline_risk    4940
engagement_gap              3415
general_review              2860
page_one_aged                973
thin_visible                  82
strong_visible                36

Top 10 preview:
 rank  final_refresh_score              suggested_action       archetype confidence                                                                                                                        

## 2. Intended use and limits

### Intended use (who / for what)

- **Who:** an SEO or content editor with limited review capacity.
- **For what:** open the top of the ranked queue first, read the reason codes, then decide
  whether to refresh copy, fix title/meta (CTR), expand thin pages, protect winners, or monitor.
- **Success measure that matches the job:** Precision@K on a holdout — *of the pages we put
  at the top, how many look like review candidates under our proxy label?*

### Where validity stops

| Claim boundary | What we measured | What we did **not** measure |
|---|---|---|
| Ranking quality | Client-holdout Precision@K / ROC AUC on a snapshot decline proxy | Future-window recovery after an edit |
| Transfer | Unseen clients in this 32-client starter slice | Other verticals, languages, or the full warehouse |
| Signals | CTR-gap among visible pages (CONFIRMED); deep-stale alone (OPPOSITE) | That age *causes* decline |
| Product role | Decision-support triage list | Auto-publish, auto-prune, or Google-algorithm prediction |

### Cost / value thinking (practical, non-production)

- **Editor time is the scarce resource.** High-confidence, high-impression rows repay review time;
  low-demand `monitor` rows usually do not.
- **Cost of a false positive (top of queue):** wasted edit hours on a page that was seasonal,
  consolidated, or already fixed after the snapshot.
- **Cost of a false negative:** a visible page-one URL keeps sliding while capacity is spent on noise.
- **Rule of thumb for this playbook:** review **high** confidence first; sample **medium**;
  skip bulk automation on **low** / `low_demand_noise`.
- **Compute cost** of re-scoring this starter slice is cheap; the expensive part is human review
  and any live experiment that would measure true recovery.

In [2]:
w05 = json.loads(W05_PATH.read_text(encoding="utf-8")) if W05_PATH.exists() else {}
w06 = json.loads(W06_PATH.read_text(encoding="utf-8")) if W06_PATH.exists() else {}

print("=== Intended-use receipts (from prior weeks + this run) ===")
if w05:
    rf_row = next(r for r in w05["comparison"] if r["method"] == "random_forest")
    base_row = next(r for r in w05["comparison"] if r["method"] == "week4_ctr_gap_baseline")
    print(
        f"W05 client-holdout RF Precision@50={rf_row['precision@50']:.3f} "
        f"vs CTR-gap baseline={base_row['precision@50']:.3f} "
        f"(test n={w05['test_rows']}, base rate={w05['base_rate_test']:.3f})"
    )
    print("W05 claim:", w05.get("claim"))
if w06:
    print("W06 safe claim:", w06.get("safe_claim"))

print(
    f"\nThis-run holdout check: Precision@50={p50:.3f} ROC AUC={roc:.3f} "
    f"(must stay decision-support language)"
)

# Cost/value sketch: capacity bands by confidence × demand
bands = (
    queue.assign(high_demand=queue["impressions_90d"] >= 500)
    .groupby(["confidence", "high_demand"], observed=True)
    .size()
    .rename("n")
    .reset_index()
)
print("\nCapacity bands (confidence × impressions>=500):")
print(bands.to_string(index=False))

high_hi = queue[(queue["confidence"] == "high") & (queue["impressions_90d"] >= 500)]
print(
    f"\nSuggested first-pass capacity: {len(high_hi):,} high-confidence + high-demand pages "
    f"(median impressions={high_hi['impressions_90d'].median():.0f})."
)
print("Not intended for: auto-publishing edits, auto-deleting URLs, or claiming causal SEO lift.")

=== Intended-use receipts (from prior weeks + this run) ===
W05 client-holdout RF Precision@50=0.900 vs CTR-gap baseline=0.300 (test n=2325, base rate=0.391)
W05 claim: Model scores are decision-support ranks on a snapshot decline proxy; not causal evidence that a refresh recovers traffic.
W06 safe claim: On a client-holdout of the starter snapshot (test n=2,325, 6 unseen clients), the random forest measured Precision@50=0.900 and ROC AUC=0.764 against the declining-trend proxy (test base rate=0.391; random-split Precision@50 was 0.960). That is a directional, decision-support ranking signal for which pages look worth reviewing first -- not causal evidence that a refresh recovers traffic, and not a claim about Google's algorithm.

This-run holdout check: Precision@50=0.900 ROC AUC=0.764 (must stay decision-support language)

Capacity bands (confidence × impressions>=500):
confidence  high_demand    n
      high         True 3460
       low        False 9125
       low         True 5875

## 3. Human review + the no-go list

### What a person must check before acting

1. **Open the live page** — does the reason code still match what you see (title/meta, thin copy, outdated facts)?
2. **Rule out look-alikes** — consolidation (sibling URL absorbed demand), seasonality, SERP/AI click theft
   (impressions/position hold while CTR falls), or low-volume noise.
3. **Challenge age-only stories** — on this snapshot, deep-stale alone was **not** a clean decline signal.
4. **Confirm demand** — if impressions are tiny, prefer `monitor` over a heavy rewrite.
5. **Protect winners** — `protect` / strong visible pages should not be churned because a score looked busy.

### What should NOT be automated (no-go)

| No-go | Why |
|---|---|
| Auto-publish title/meta/body changes from the score | Observational ranking ≠ editorial judgment or brand voice |
| Auto-prune / 301 / unindex from the queue | Irreversible SEO actions need human + business context |
| Treat `declining_with_demand` as proof a refresh will recover traffic | Label is a snapshot trend proxy, not a causal design |
| Feed `trend_pct` / `trend_direction` back as model features | Label trap — validation becomes fake-perfect |
| Ship client names, raw queries, domains, or private URLs in exports | Public-safety / data-use rules |
| Production "set and forget" scoring without monitoring | Drift, seasonality, and inventory mix change |
| Age-only auto-refresh triggers | Staleness verdict was OPPOSITE on this starter slice |

In [3]:
REVIEW_COLS = [
    "rank",
    "content_id",
    "suggested_action",
    "archetype",
    "confidence",
    "reason_codes",
    "impressions_90d",
    "avg_position",
    "ctr",
    "days_since_last_update",
    "is_declining_label",
]

top20 = queue.head(20)[REVIEW_COLS].copy()
print("Top-20 human-review sheet (pseudonym IDs only):")
print(top20.to_string(index=False))

print("\n--- Skeptic prompts for the top 5 ---")
for _, row in top20.head(5).iterrows():
    print(f"#{int(row['rank'])} action={row['suggested_action']} archetype={row['archetype']}")
    print(f"  reasons: {row['reason_codes']}")
    print(
        "  check: consolidation / seasonality / SERP feature / already-fixed meta / "
        "age-only story (staleness was OPPOSITE here)?"
    )

nogo = [
    "Do not auto-publish edits from this queue",
    "Do not auto-prune or redirect from this queue",
    "Do not claim refresh recovers traffic from this observational score",
    "Do not use trend_pct/trend_direction as model features",
    "Do not export client names, domains, raw queries, or private URLs",
    "Do not trigger refreshes from age alone on this snapshot",
]
print("\nNo-go checklist:")
for item in nogo:
    print(" [ ]", item)

Top-20 human-review sheet (pseudonym IDs only):
 rank           content_id              suggested_action       archetype confidence                                                                                                                                                         reason_codes  impressions_90d  avg_position  ctr  days_since_last_update  is_declining_label
    1 content_92c274459342 refresh_and_review_engagement  engagement_gap       high                                           declining_with_demand|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|engagement_review_candidate            14099          26.0 0.45                     106                   1
    2 content_87f1ffe0bedb        refresh_and_review_ctr visible_ctr_gap       high                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate             8225           1.7 0.23          

## 4. Monitoring / retrain triggers

This playbook is a **research prototype**, not a production service. Still define what would
tell you the recommendations went stale:

| Trigger | Threshold idea | Response |
|---|---|---|
| Holdout Precision@50 drops vs last receipt | Fall **>0.10** absolute vs W05/W06 RF number on a comparable split | Re-audit features/label; do not trust the queue until fixed |
| Base rate shift | Declining-label (or future-window) rate moves **>0.10** vs prior snapshot | Recalibrate confidence bands; retrain |
| Action mix collapse | One action &gt;90% of top-100 (score gaming / inventory shock) | Inspect reason-code rules; freeze automation |
| Hand-review reject rate | Editor rejects **≥40%** of top-20 in a weekly sample | Tighten high-confidence gates; revisit CTR/age rules |
| Feature / schema drift | New missingness by `content_type`, position zeros surge, rate columns change scale | Stop scoring; fix contract first |
| Time passed | **≥90 days** since last validated retrain on a comparable window | Scheduled refresh of model + baseline blend |

Preferred next validation (stronger than this starter proxy): warehouse **past→future** decline
label with sealed final month — retrain only after that design is honest.

In [4]:
# Monitoring dashboard numbers for the paper / ops log
rf_p50_receipt = None
if w05:
    rf_p50_receipt = next(r for r in w05["comparison"] if r["method"] == "random_forest")["precision@50"]

top100_action_share = float(queue.head(100)["suggested_action"].value_counts(normalize=True).iloc[0])
top100_dominant = queue.head(100)["suggested_action"].value_counts().index[0]

triggers = {
    "precision_at_50_this_run": round(p50, 4),
    "precision_at_50_w05_receipt": rf_p50_receipt,
    "precision_drop_abs_vs_w05": None
    if rf_p50_receipt is None
    else round(float(rf_p50_receipt) - p50, 4),
    "retrain_if_precision_drop_gt": 0.10,
    "declining_label_rate": round(float(y.mean()), 4),
    "retrain_if_base_rate_shift_gt": 0.10,
    "top100_dominant_action": top100_dominant,
    "top100_dominant_share": round(top100_action_share, 4),
    "freeze_if_top100_one_action_share_gt": 0.90,
    "hand_review_reject_rate_alert_gt": 0.40,
    "scheduled_retrain_days": 90,
}

print("Monitoring snapshot:")
for k, v in triggers.items():
    print(f"  {k}: {v}")

alert_precision = (
    triggers["precision_drop_abs_vs_w05"] is not None
    and triggers["precision_drop_abs_vs_w05"] > triggers["retrain_if_precision_drop_gt"]
)
alert_mix = triggers["top100_dominant_share"] > triggers["freeze_if_top100_one_action_share_gt"]
print("\nAlert flags now:")
print("  precision_drop:", alert_precision)
print("  action_mix_collapse:", alert_mix)
print("  (hand-review reject rate: measure offline when editors sample top-20)")

Monitoring snapshot:
  precision_at_50_this_run: 0.9
  precision_at_50_w05_receipt: 0.9
  precision_drop_abs_vs_w05: 0.0
  retrain_if_precision_drop_gt: 0.1
  declining_label_rate: 0.5421
  retrain_if_base_rate_shift_gt: 0.1
  top100_dominant_action: refresh_and_review_ctr
  top100_dominant_share: 0.63
  freeze_if_top100_one_action_share_gt: 0.9
  hand_review_reject_rate_alert_gt: 0.4
  scheduled_retrain_days: 90

Alert flags now:
  precision_drop: False
  action_mix_collapse: False
  (hand-review reject rate: measure offline when editors sample top-20)


## 5. Exports for the paper

Write the ranked queue and reusable figures under `work/`.

| Artifact | Path | Git? |
|---|---|---|
| Full ranked queue | `work/outputs/w07_action_playbook_queue.csv` | **No** (CI leak-guard; regenerate by running this notebook) |
| Metrics / playbook receipts | `work/outputs/w07_action_playbook_metrics.json` | **Yes** |
| Action-mix figure | `work/figures/w07_action_mix.svg` | **Yes** |
| Archetype→action figure | `work/figures/w07_archetype_action.svg` | **Yes** |
| Top reason codes | `work/figures/w07_top_reason_codes.svg` | **Yes** |

Next week’s paper recommendations section should cite these files — not reinvent the queue.

In [5]:
def escape_xml(value: str) -> str:
    return (
        str(value)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
    )


def simple_svg_bar_chart(title, labels, values, path, color="#426B69", width=960, height=520):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    labels = [str(x) for x in labels]
    values = [float(v) for v in values]
    max_value = max(values) if values else 1.0
    max_value = max(max_value, 1.0)
    margin_left, margin_right, margin_top, margin_bottom = 220, 40, 70, 50
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    bar_gap = 10
    bar_height = max(14, (plot_height - bar_gap * max(len(values) - 1, 0)) / max(len(values), 1))
    lines = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{width / 2}" y="34" text-anchor="middle" font-family="Arial" font-size="22" fill="#16232a">{escape_xml(title)}</text>',
    ]
    for i, (label, value) in enumerate(zip(labels, values)):
        y = margin_top + i * (bar_height + bar_gap)
        bar_width = (value / max_value) * plot_width
        lines.append(
            f'<text x="{margin_left - 12}" y="{y + bar_height * 0.65:.1f}" text-anchor="end" '
            f'font-family="Arial" font-size="13" fill="#27343b">{escape_xml(label[:36])}</text>'
        )
        lines.append(
            f'<rect x="{margin_left}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" '
            f'fill="{color}" rx="4"/>'
        )
        lines.append(
            f'<text x="{margin_left + bar_width + 8:.1f}" y="{y + bar_height * 0.65:.1f}" '
            f'font-family="Arial" font-size="13" fill="#27343b">{value:,.3g}</text>'
        )
    lines.append("</svg>")
    path.write_text("\n".join(lines), encoding="utf-8")


export_cols = [
    "rank",
    "content_id",
    "client_id",
    "final_refresh_score",
    "model_probability",
    "baseline_refresh_score",
    "confidence",
    "suggested_action",
    "archetype",
    "reason_codes",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "content_type",
    "position_tier",
    "freshness_tier",
]
ranked = queue[export_cols].copy()
ranked.to_csv(QUEUE_PATH, index=False)

action_counts = ranked["suggested_action"].value_counts()
simple_svg_bar_chart(
    "W07 suggested action mix",
    action_counts.index.tolist(),
    action_counts.values.tolist(),
    FIG_DIR / "w07_action_mix.svg",
    color="#426B69",
)

arch_action = (
    ranked.groupby(["archetype", "suggested_action"], observed=True)
    .size()
    .rename("n")
    .reset_index()
    .sort_values("n", ascending=False)
    .head(12)
)
arch_labels = [f"{a} → {s}" for a, s in zip(arch_action["archetype"], arch_action["suggested_action"])]
simple_svg_bar_chart(
    "W07 archetype → action (top pairs)",
    arch_labels,
    arch_action["n"].tolist(),
    FIG_DIR / "w07_archetype_action.svg",
    color="#4E79A7",
)

reason_counts = {}
for text in ranked["reason_codes"]:
    for reason in str(text).split("|"):
        if reason:
            reason_counts[reason] = reason_counts.get(reason, 0) + 1
top_reasons = pd.Series(reason_counts).sort_values(ascending=False).head(12)
simple_svg_bar_chart(
    "W07 top reason codes",
    top_reasons.index.tolist(),
    top_reasons.values.tolist(),
    FIG_DIR / "w07_top_reason_codes.svg",
    color="#8C6BB1",
)

fi = pd.Series(rf.feature_importances_, index=NUMERIC_FEATURES).sort_values(ascending=False)
metrics = {
    "lane": "Lane 2: Refresh / Content Opportunity Scoring",
    "notebook": "work/notebooks/w07_action_playbook.ipynb",
    "random_state": RANDOM_STATE,
    "split": "client_holdout_for_validation_metrics",
    "rows_scored": int(len(ranked)),
    "test_rows": int(len(test_idx)),
    "test_clients": int(n_test_clients),
    "base_rate_all": round(float(y.mean()), 4),
    "holdout_metrics": {
        "precision@10": round(p10, 4),
        "precision@50": round(p50, 4),
        "roc_auc": round(roc, 4),
        "avg_precision": round(ap, 4),
    },
    "score_formula": "100 * (0.70 * model_probability + 0.30 * normalized_baseline)",
    "signal_verdicts_carried_forward": {
        "staleness": "OPPOSITE",
        "ctr_vs_position": "CONFIRMED",
    },
    "action_counts": {k: int(v) for k, v in action_counts.items()},
    "archetype_counts": {k: int(v) for k, v in ranked["archetype"].value_counts().items()},
    "confidence_counts": {k: int(v) for k, v in ranked["confidence"].value_counts().items()},
    "top_features": {k: round(float(v), 4) for k, v in fi.head(8).items()},
    "monitoring_triggers": triggers,
    "exports": {
        "queue_csv": str(QUEUE_PATH).replace("\\", "/"),
        "metrics_json": str(METRICS_PATH).replace("\\", "/"),
        "figures": [
            "work/figures/w07_action_mix.svg",
            "work/figures/w07_archetype_action.svg",
            "work/figures/w07_top_reason_codes.svg",
        ],
    },
    "safe_claim": (
        "On this starter snapshot, the playbook ranks pages for human refresh review using a "
        f"blended RF+baseline score (client-holdout Precision@50={p50:.3f}). "
        "It is directional decision-support with reason codes — not causal evidence that a "
        "refresh recovers traffic, and not a Google-algorithm prediction."
    ),
    "no_go": nogo,
}
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print(f"Wrote {QUEUE_PATH} ({len(ranked):,} rows)")
print(f"Wrote {METRICS_PATH}")
print(f"Wrote figures under {FIG_DIR}/")
print("\nSafe claim:")
print(metrics["safe_claim"])
print("\nTop-5 export preview:")
print(
    ranked.head(5)[
        ["rank", "final_refresh_score", "suggested_action", "archetype", "confidence", "reason_codes"]
    ].to_string(index=False)
)

Wrote work\outputs\w07_action_playbook_queue.csv (30,000 rows)
Wrote work\outputs\w07_action_playbook_metrics.json
Wrote figures under work\figures/

Safe claim:
On this starter snapshot, the playbook ranks pages for human refresh review using a blended RF+baseline score (client-holdout Precision@50=0.900). It is directional decision-support with reason codes — not causal evidence that a refresh recovers traffic, and not a Google-algorithm prediction.

Top-5 export preview:
 rank  final_refresh_score              suggested_action       archetype confidence                                                                                                                                                         reason_codes
    1            90.719065 refresh_and_review_engagement  engagement_gap       high                                           declining_with_demand|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|engagement_review_candidate
    2            90.186

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Queue exported to `work/outputs/` (CSV regenerates; metrics JSON + figures committed)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.